# BBKNN Batch Integration Pipeline

**BBKNN (Batch Balanced k-Nearest Neighbors) Integration**

- Graph-based fast batch integration
- Highly variable gene selection
- Batch integration quality assessment
- Multi-resolution clustering
- Enhanced visualization

**Author:** Clinical-Bioinformatics Team  
**Date:** 2025-11-01  
**Version:** v1.0

## 1. Import Libraries and Setup

In [ ]:
import sys
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import time
from datetime import datetime
import json

import scanpy as sc
import bbknn
import matplotlib.pyplot as plt
from scipy.stats import entropy
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')

# Check versions
print(f"scanpy: {sc.__version__}")
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 2. Configuration Parameters

In [ ]:
# ========== Input/Output Configuration ==========
INPUT_H5AD_PATH = "/home/h2048/data/R/1124/merged_object_sc.h5ad"
OUTPUT_DIR = "/home/h2048/data/py/1225/bbknn_output_optimized"
OVERWRITE_EXISTING = True

# ========== BBKNN Integration Configuration ==========
BATCH_KEY = "dataset"  # Batch variable name

# BBKNN core parameters
BBKNN_NEIGHBORS_WITHIN_BATCH = 5  # Number of neighbors within each batch (recommended 3-5)
BBKNN_N_PCS = 50  # Number of principal components to use

# ========== Highly Variable Genes Selection ==========
USE_HVG = True  # Whether to select highly variable genes
N_TOP_GENES = 3000  # Number of highly variable genes to keep
HVG_FLAVOR = "seurat_v3"  # HVG selection method

# ========== Gene Filtering Configuration ==========
MIN_CELLS_PER_GENE = 3

# ========== Normalization Configuration ==========
NORMALIZE_TOTAL = True  # Whether to perform total count normalization
TARGET_SUM = 1e4  # Target sum for normalization
LOG_TRANSFORM = True  # Whether to perform log transformation
SCALE_DATA = True  # Whether to scale data
MAX_VALUE = 10  # Maximum value for scaling

# ========== PCA Configuration ==========
N_PCS = 50  # Number of principal components to compute

# ========== Dimensionality Reduction and Visualization ==========
RUN_UMAP = True
UMAP_MIN_DIST = 0.5
UMAP_N_NEIGHBORS = 100  # UMAP neighbors (will be overridden by BBKNN neighbor graph)

# ========== Clustering Configuration (Multi-resolution) ==========
RUN_CLUSTERING = True
LEIDEN_RESOLUTIONS = [0.2, 0.4, 0.6, 0.8]  # Multiple resolutions
DEFAULT_RESOLUTION = 0.4  # Default resolution to use

# ========== Batch Integration Evaluation ==========
EVALUATE_INTEGRATION = False  # Whether to evaluate batch integration quality

# ========== Visualization Variables ==========
VISUALIZATION_VARS = [
    "dataset",
    "Annotation",
    "tissue_sampling_method",
]

# ========== Enhanced Visualization ==========
GENERATE_FACET_PLOTS = True  # Whether to generate faceted plots by batch

VERBOSE = True

print("Configuration loaded successfully")
print(f"Input file: {INPUT_H5AD_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Batch key: {BATCH_KEY}")
print(f"BBKNN neighbors within batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")

## 3. Create Output Directory

In [ ]:
# Create output directory
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

fig_dir = output_dir / "figures"
fig_dir.mkdir(exist_ok=True)

print(f"Output directory created: {output_dir}")
print(f"Figures directory: {fig_dir}")

# Set scanpy figure directory
sc.settings.figdir = fig_dir

# Record start time
start_time = time.time()
print(f"\nAnalysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 4. Load Data

In [ ]:
print("="*70)
print("Step 1: Loading Data")
print("="*70)

print(f"\nReading file: {INPUT_H5AD_PATH}")

if not Path(INPUT_H5AD_PATH).exists():
    raise FileNotFoundError(f"File not found: {INPUT_H5AD_PATH}")

# Read data
adata = sc.read_h5ad(INPUT_H5AD_PATH)

print(f"Data loaded successfully")
print(f"   Cells: {adata.n_obs:,}")
print(f"   Genes: {adata.n_vars:,}")

# Display obs columns
print(f"\nAvailable metadata columns:")
for col in adata.obs.columns:
    n_unique = adata.obs[col].nunique()
    print(f"   - {col}: {n_unique} unique values")

# Check batch key
if BATCH_KEY not in adata.obs.columns:
    raise ValueError(f"Batch key '{BATCH_KEY}' not found in adata.obs")

# Display batch distribution
print(f"\nBatch distribution (key: {BATCH_KEY}):")
batch_counts = adata.obs[BATCH_KEY].value_counts().sort_index()
for batch, count in batch_counts.items():
    pct = count / adata.n_obs * 100
    print(f"   {batch}: {count:,} cells ({pct:.1f}%)")

## 5. Preprocessing for BBKNN

In [ ]:
print("\n" + "="*70)
print("Step 2: Preprocessing for BBKNN")
print("="*70)

# Save original counts if not already saved
if 'counts' not in adata.layers:
    print("\nSaving raw counts to layers['counts']...")
    adata.layers['counts'] = adata.X.copy()
else:
    print("\nRaw counts already saved in layers['counts']")

In [ ]:
# Gene filtering
print(f"\nGene filtering (min_cells={MIN_CELLS_PER_GENE})...")
n_genes_before = adata.n_vars
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
n_genes_after = adata.n_vars

print(f"   Genes before: {n_genes_before:,}")
print(f"   Genes after: {n_genes_after:,}")
print(f"   Removed: {n_genes_before - n_genes_after:,}")

In [ ]:
# Normalization
if NORMALIZE_TOTAL:
    print(f"\nNormalizing total counts (target_sum={TARGET_SUM})...")
    sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
    print("   Normalization completed")

In [ ]:
# Log transformation
if LOG_TRANSFORM:
    print("\nApplying log1p transformation...")
    sc.pp.log1p(adata)
    print("   Log transformation completed")

In [ ]:
# Highly variable genes selection
if USE_HVG:
    print(f"\nSelecting highly variable genes (n={N_TOP_GENES})...")
    print(f"   Method: {HVG_FLAVOR}")
    
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_TOP_GENES,
        flavor=HVG_FLAVOR,
        batch_key=BATCH_KEY,
        subset=False  # Don't subset yet, keep all gene info
    )
    
    n_hvg = adata.var['highly_variable'].sum()
    print(f"   Selected {n_hvg} highly variable genes")
    
    # Create subset for downstream analysis
    adata_hvg = adata[:, adata.var['highly_variable']].copy()
    print(f"   Final genes for downstream analysis: {adata_hvg.n_vars:,}")
else:
    adata_hvg = adata.copy()
    print("\nSkipping HVG selection, using all genes")

In [ ]:
# Scale data (only on HVG)
if SCALE_DATA:
    print(f"\nScaling data (max_value={MAX_VALUE})...")
    sc.pp.scale(adata_hvg, max_value=MAX_VALUE)
    print("   Scaling completed")

print("\nPreprocessing completed")
print(f"   Final dimensions: {adata_hvg.n_obs:,} cells x {adata_hvg.n_vars:,} genes")

## 6. Analyze Unintegrated Data (Baseline)

In [ ]:
print("\n" + "="*70)
print("Step 3: Analyzing Unintegrated Data (Baseline)")
print("="*70)

# Create temporary copy for unintegrated analysis
adata_temp = adata.copy()

# Use preprocessed data
if 'counts' in adata_temp.layers:
    print("\nUsing preprocessed data...")
    adata_temp.X = adata_temp.layers['counts'].copy()
    
    # Normalize
    if NORMALIZE_TOTAL:
        sc.pp.normalize_total(adata_temp, target_sum=TARGET_SUM)
    if LOG_TRANSFORM:
        sc.pp.log1p(adata_temp)

# If using HVG, subset to highly variable genes
if USE_HVG and 'highly_variable' in adata_temp.var:
    print("Subsetting to highly variable genes...")
    adata_temp = adata_temp[:, adata_temp.var['highly_variable']].copy()

# Scale and PCA
print("\nRunning PCA on unintegrated data...")
if SCALE_DATA:
    sc.pp.scale(adata_temp, max_value=MAX_VALUE)
sc.tl.pca(adata_temp, n_comps=50)

# Standard neighbors and UMAP (without BBKNN)
print("Computing standard neighbors and UMAP...")
sc.pp.neighbors(adata_temp, n_neighbors=UMAP_N_NEIGHBORS, use_rep="X_pca")
sc.tl.umap(adata_temp, min_dist=UMAP_MIN_DIST)

print("   Unintegrated analysis completed")

In [ ]:
# Visualize unintegrated data
print("\nGenerating unintegrated visualizations...")

# UMAP by batch
if BATCH_KEY in adata_temp.obs.columns:
    sc.pl.umap(
        adata_temp,
        color=BATCH_KEY,
        show=False,
        title='UMAP - Unintegrated (by batch)',
        save='_unintegrated_batch.png'
    )
    print(f"   Saved: {fig_dir}/umap_unintegrated_batch.png")

# UMAP by cell type (if available)
if 'cell_type' in adata_temp.obs.columns:
    sc.pl.umap(
        adata_temp,
        color='cell_type',
        show=False,
        title='UMAP - Unintegrated (by cell type)',
        save='_unintegrated_celltype.png'
    )
    print(f"   Saved: {fig_dir}/umap_unintegrated_celltype.png")

# Clean up temporary object
del adata_temp
print("\nUnintegrated baseline analysis completed")

## 7. PCA on Preprocessed Data

In [ ]:
print("\n" + "="*70)
print("Step 4: Running PCA")
print("="*70)

print(f"\nRunning PCA (n_comps={N_PCS})...")
sc.tl.pca(adata_hvg, n_comps=N_PCS, svd_solver='arpack')

# Calculate explained variance ratio
var_ratio = adata_hvg.uns['pca']['variance_ratio']
cumsum_var = np.cumsum(var_ratio)

print(f"   PC1-10 explained variance: {cumsum_var[9]:.2%}")
print(f"   PC1-20 explained variance: {cumsum_var[19]:.2%}")
print(f"   PC1-50 explained variance: {cumsum_var[49]:.2%}")
print("\nPCA completed")

## 8. BBKNN Batch Integration

In [ ]:
print("\n" + "="*70)
print("Step 5: BBKNN Batch Integration")
print("="*70)

print("\nBBKNN parameters:")
print(f"   batch_key: {BATCH_KEY}")
print(f"   neighbors_within_batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")
print(f"   n_pcs: {BBKNN_N_PCS}")

print("\nRunning BBKNN...")
print("   BBKNN constructs batch-balanced k-nearest neighbor graph")
print("   This may take a few minutes depending on data size...")

bbknn_start = time.time()

# Run BBKNN
# BBKNN will directly modify adata neighbors information
bbknn.bbknn(
    adata_hvg,
    batch_key=BATCH_KEY,
    neighbors_within_batch=BBKNN_NEIGHBORS_WITHIN_BATCH,
    n_pcs=BBKNN_N_PCS,
    copy=False  # Modify adata in place
)

bbknn_time = time.time() - bbknn_start
print(f"   BBKNN completed in {bbknn_time:.1f} seconds")

print("\n   Batch-balanced neighbor graph constructed")
print("   Ready for downstream analysis (UMAP, clustering)")

## 9. UMAP Dimensionality Reduction

In [ ]:
print("\n" + "="*70)
print("Step 6: UMAP and Clustering")
print("="*70)

if RUN_UMAP:
    print("\nComputing UMAP...")
    print(f"   min_dist: {UMAP_MIN_DIST}")
    print("   Using BBKNN neighbor graph")
    
    # BBKNN has already computed neighbors, directly use UMAP
    sc.tl.umap(adata_hvg, min_dist=UMAP_MIN_DIST)
    print("   UMAP completed")
else:
    print("\nSkipping UMAP (RUN_UMAP=False)")

## 10. Multi-resolution Clustering

In [ ]:
if RUN_CLUSTERING:
    print("\nRunning multi-resolution Leiden clustering...")
    print(f"   Resolutions: {LEIDEN_RESOLUTIONS}")
    
    for res in LEIDEN_RESOLUTIONS:
        key = f'leiden_bbknn_res{res}'
        sc.tl.leiden(adata_hvg, resolution=res, key_added=key)
        n_clusters = adata_hvg.obs[key].nunique()
        print(f"   Resolution {res}: {n_clusters} clusters")
    
    # Set default clustering
    default_key = f'leiden_bbknn_res{DEFAULT_RESOLUTION}'
    if default_key in adata_hvg.obs.columns:
        adata_hvg.obs['leiden_bbknn'] = adata_hvg.obs[default_key]
        print(f"\n   Default clustering: {default_key}")
else:
    print("\nSkipping clustering (RUN_CLUSTERING=False)")

## 11. Basic Visualizations

In [ ]:
print("\n" + "="*70)
print("Step 7: Generating Visualizations")
print("="*70)

if not RUN_UMAP:
    print("Skipping visualizations (UMAP not computed)")
else:
    # Basic UMAP plots
    print("\nGenerating UMAP plots...")
    for var in VISUALIZATION_VARS:
        if var in adata_hvg.obs.columns:
            print(f"   - UMAP colored by {var}")
            sc.pl.umap(
                adata_hvg,
                color=var,
                show=False,
                title=f'UMAP - {var}',
                save=f'_{var}.png'
            )
    
    print(f"\nFigures saved to: {fig_dir}")

In [ ]:
# Clustering results visualization
if RUN_CLUSTERING and RUN_UMAP:
    print("\nGenerating clustering plots...")
    for res in LEIDEN_RESOLUTIONS:
        key = f'leiden_bbknn_res{res}'
        if key in adata_hvg.obs.columns:
            print(f"   - UMAP colored by {key}")
            sc.pl.umap(
                adata_hvg,
                color=key,
                show=False,
                title=f'UMAP - Leiden (res={res})',
                save=f'_{key}.png'
            )

## 12. Facet Plots by Batch

In [ ]:
# Facet plots by batch
if GENERATE_FACET_PLOTS and BATCH_KEY in adata_hvg.obs.columns and RUN_UMAP:
    print("\nGenerating facet plots...")
    batches = adata_hvg.obs[BATCH_KEY].unique()
    
    if len(batches) <= 10:
        try:
            print(f"   - Facet plot by {BATCH_KEY}")
            
            n_batches = len(batches)
            n_cols = min(3, n_batches)
            n_rows = (n_batches + n_cols - 1) // n_cols
            
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
            if n_batches == 1:
                axes = [axes]
            else:
                axes = axes.flatten()
            
            color_var = 'leiden_bbknn' if 'leiden_bbknn' in adata_hvg.obs.columns else None
            
            for i, batch in enumerate(sorted(batches)):
                adata_batch = adata_hvg[adata_hvg.obs[BATCH_KEY] == batch]
                sc.pl.umap(
                    adata_batch,
                    color=color_var,
                    ax=axes[i],
                    show=False,
                    title=f'{batch}'
                )
            
            # Hide extra subplots
            for i in range(n_batches, len(axes)):
                axes[i].axis('off')
            
            plt.tight_layout()
            plt.savefig(fig_dir / f'umap_facet_by_{BATCH_KEY}.png', dpi=150, bbox_inches='tight')
            plt.close()
            
            print(f"   Saved: {fig_dir}/umap_facet_by_{BATCH_KEY}.png")
            
        except Exception as e:
            print(f"   Warning: Facet plot failed - {e}")
    else:
        print(f"   Skipping facet plots (too many batches: {len(batches)})")

## 13. Evaluate Integration Quality (Optional)

In [ ]:
print("\n" + "="*70)
print("Step 8: Evaluating Integration Quality")
print("="*70)

eval_results = {}

if EVALUATE_INTEGRATION:
    # Silhouette score (batch) - using PCA space
    print("\nComputing silhouette scores...")
    
    if BATCH_KEY in adata_hvg.obs.columns and 'X_pca' in adata_hvg.obsm:
        batch_labels = adata_hvg.obs[BATCH_KEY].astype('category').cat.codes
        
        # Evaluate using PCA space
        sil_batch = silhouette_score(adata_hvg.obsm['X_pca'][:, :BBKNN_N_PCS], batch_labels)
        eval_results['silhouette_batch'] = float(sil_batch)
        print(f"   Silhouette (batch): {sil_batch:.4f}")
        print(f"   Interpretation: closer to 0 = better batch mixing")
    
    # If cell type information is available, compute biological conservation
    if 'cell_type' in adata_hvg.obs.columns and 'X_pca' in adata_hvg.obsm:
        print("\nComputing biological conservation...")
        cell_type_labels = adata_hvg.obs['cell_type'].astype('category').cat.codes
        
        sil_bio = silhouette_score(adata_hvg.obsm['X_pca'][:, :BBKNN_N_PCS], cell_type_labels)
        eval_results['silhouette_biology'] = float(sil_bio)
        print(f"   Silhouette (cell type): {sil_bio:.4f}")
        print(f"   Interpretation: closer to 1 = better biological separation")
    
    # Compute batch mixing entropy
    if 'leiden_bbknn' in adata_hvg.obs.columns:
        print("\nComputing batch mixing metrics...")
        
        # For each cluster, compute entropy of batch distribution
        clusters = adata_hvg.obs['leiden_bbknn'].unique()
        entropies = []
        
        for cluster in clusters:
            cluster_mask = adata_hvg.obs['leiden_bbknn'] == cluster
            batch_dist = adata_hvg.obs.loc[cluster_mask, BATCH_KEY].value_counts(normalize=True)
            ent = entropy(batch_dist)
            entropies.append(ent)
        
        mean_entropy = np.mean(entropies)
        eval_results['mean_cluster_batch_entropy'] = float(mean_entropy)
        print(f"   Mean cluster batch entropy: {mean_entropy:.4f}")
        print(f"   Interpretation: higher = better batch mixing within clusters")
    
    # Save results
    eval_path = output_dir / "integration_evaluation.json"
    with open(eval_path, 'w') as f:
        json.dump(eval_results, f, indent=2)
    
    print(f"\nEvaluation results saved to: {eval_path}")
else:
    print("\nSkipping integration evaluation (EVALUATE_INTEGRATION=False)")

## 14. Transfer Results to Full Dataset

In [ ]:
print("\n" + "="*70)
print("Step 9: Transferring Results to Full Dataset")
print("="*70)

print("\nTransferring results to full dataset...")

# Transfer embeddings
for key in ['X_pca', 'X_umap']:
    if key in adata_hvg.obsm:
        adata.obsm[key] = adata_hvg.obsm[key]
        print(f"   Transferred {key}")

# Transfer clustering results
for col in adata_hvg.obs.columns:
    if col.startswith('leiden_bbknn'):
        adata.obs[col] = adata_hvg.obs[col]
        print(f"   Transferred {col}")

# Copy neighbors information
if 'neighbors' in adata_hvg.uns:
    adata.uns['neighbors'] = adata_hvg.uns['neighbors']
    print("   Transferred neighbors information")

if 'connectivities' in adata_hvg.obsp:
    adata.obsp['connectivities'] = adata_hvg.obsp['connectivities']
    print("   Transferred connectivities")

if 'distances' in adata_hvg.obsp:
    adata.obsp['distances'] = adata_hvg.obsp['distances']
    print("   Transferred distances")

print("\nTransfer completed")

## 15. Save Integrated Data

In [ ]:
print("\n" + "="*70)
print("Step 10: Saving Integrated Data")
print("="*70)

print("\nSaving integrated data...")
final_path = output_dir / "adata_bbknn_integrated.h5ad"

print("   Using gzip compression (level 9)...")
adata.write_h5ad(final_path, compression='gzip', compression_opts=9)

file_size = final_path.stat().st_size / (1024**3)
print(f"   Saved: {final_path} ({file_size:.2f} GB)")

## 16. Generate Summary Report

In [ ]:
print("\n" + "="*70)
print("Step 11: Generating Summary Report")
print("="*70)

# Calculate total processing time
total_time = time.time() - start_time

# Generate report
report = []
report.append("="*70)
report.append("BBKNN Batch Integration - Analysis Summary")
report.append("="*70)
report.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
report.append(f"Total processing time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
report.append("")

# Data overview
report.append("[ Data Overview ]")
report.append(f"  Cells: {adata.n_obs:,}")
report.append(f"  Genes: {adata.n_vars:,}")
report.append("")

# Batch information
if BATCH_KEY in adata.obs.columns:
    report.append(f"[ Batch Distribution (key: {BATCH_KEY}) ]")
    batch_counts = adata.obs[BATCH_KEY].value_counts().sort_index()
    for batch, count in batch_counts.items():
        pct = count / adata.n_obs * 100
        report.append(f"  {batch}: {count:,} cells ({pct:.1f}%)")
    report.append(f"  Total batches: {len(batch_counts)}")
    report.append("")

# Integration quality metrics
if eval_results:
    report.append("[ Integration Quality Metrics ]")
    if 'silhouette_batch' in eval_results:
        report.append(f"  Silhouette Score (batch): {eval_results['silhouette_batch']:.4f}")
        report.append("  (Interpretation: closer to 0 = better batch mixing)")
    if 'silhouette_biology' in eval_results:
        report.append(f"  Silhouette Score (cell type): {eval_results['silhouette_biology']:.4f}")
        report.append("  (Interpretation: closer to 1 = better biological separation)")
    if 'mean_cluster_batch_entropy' in eval_results:
        report.append(f"  Mean cluster batch entropy: {eval_results['mean_cluster_batch_entropy']:.4f}")
        report.append("  (Interpretation: higher = better batch mixing)")
    report.append("")

# Multi-resolution clustering
if RUN_CLUSTERING:
    report.append("[ Multi-resolution Clustering Results ]")
    for res in LEIDEN_RESOLUTIONS:
        key = f'leiden_bbknn_res{res}'
        if key in adata.obs.columns:
            n_clusters = adata.obs[key].nunique()
            default_marker = " (default)" if res == DEFAULT_RESOLUTION else ""
            report.append(f"  Resolution {res}: {n_clusters} clusters{default_marker}")
    report.append("")

# Available embeddings
report.append("[ Available Embeddings ]")
for key in adata.obsm.keys():
    report.append(f"  - {key}: {adata.obsm[key].shape}")
report.append("")

# BBKNN configuration
report.append("[ BBKNN Configuration ]")
report.append(f"  neighbors_within_batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")
report.append(f"  n_pcs: {BBKNN_N_PCS}")
report.append(f"  batch_key: {BATCH_KEY}")
report.append(f"  High variable genes: {USE_HVG} (n={N_TOP_GENES if USE_HVG else 'N/A'})")
report.append("")

# Output files
report.append("[ Output Files ]")
report.append(f"  - Data: {output_dir / 'adata_bbknn_integrated.h5ad'}")
report.append(f"  - Figures: {output_dir / 'figures/'}*.png")
if EVALUATE_INTEGRATION:
    report.append(f"  - Evaluation: {output_dir / 'integration_evaluation.json'}")
report.append("")

# Method notes
report.append("[ Method Notes ]")
report.append("  BBKNN (Batch Balanced k-Nearest Neighbors):")
report.append("  - Graph-based batch correction method")
report.append("  - Constructs batch-balanced neighbor graph")
report.append("  - Fast, no GPU required")
report.append("  - Suitable for well-separated batches")
report.append("")

report.append("="*70)
report.append("Analysis completed successfully")
report.append("="*70)

report_text = '\n'.join(report)

# Save report
report_path = output_dir / "analysis_summary.txt"
with open(report_path, 'w') as f:
    f.write(report_text)

print(f"\nReport saved to: {report_path}")
print("\n" + report_text)

## 17. Final Summary

In [ ]:
print("\n" + "="*70)
print("All analyses completed successfully")
print("="*70)
print(f"\nOutput directory: {output_dir}")
print(f"Data: {final_path} ({file_size:.2f} GB)")
if RUN_UMAP:
    print(f"Figures: {output_dir / 'figures/'}*.png")
if EVALUATE_INTEGRATION:
    print(f"Evaluation: {output_dir / 'integration_evaluation.json'}")
print(f"\nTotal time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")

print(f"\nKey parameters:")
print(f"   BBKNN neighbors_within_batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")
print(f"   High variable genes: {N_TOP_GENES if USE_HVG else 'Not used'}")
print(f"   Multi-resolution clustering: {LEIDEN_RESOLUTIONS}")
print()